In [1]:
# Grid reconstrucción - Autoencoder (Colab-ready)
# Ejecuta todo en Google Colab (pegar en una celda y correr).
# Recomiendo activar GPU en Runtime -> Change runtime type -> GPU.

# 1) Instalación (ejecutar una vez)
!pip install -q torch torchvision ipywidgets plotly

# Activar widgets en Colab
try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

# -------------------------------------------------------
# 2) Imports
# -------------------------------------------------------
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import math
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -------------------------------------------------------
# 3) Dataset loader (Fashion-MNIST / MNIST)
# -------------------------------------------------------
def get_dataset(name='FashionMNIST', train=True):
    transform = transforms.Compose([transforms.ToTensor()])
    if name == 'FashionMNIST':
        ds = datasets.FashionMNIST(root='./data', train=train, download=True, transform=transform)
    elif name == 'MNIST':
        ds = datasets.MNIST(root='./data', train=train, download=True, transform=transform)
    else:
        raise ValueError("Dataset desconocido")
    # Return dataset object
    return ds

# -------------------------------------------------------
# 4) Conv Autoencoder (simple, educativo)
# -------------------------------------------------------
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),  # 28->14
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), # 14->7
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32*7*7, latent_dim)
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32*7*7),
            nn.ReLU(),
            nn.Unflatten(1, (32,7,7)),
            nn.ConvTranspose2d(32,16,3,stride=2,padding=1,output_padding=1), # 7->14
            nn.ReLU(),
            nn.ConvTranspose2d(16,1,3,stride=2,padding=1,output_padding=1),  # 14->28
            nn.Sigmoid()
        )
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out
    def encode(self, x):
        return self.encoder(x)
    def decode(self, z):
        return self.decoder(z)

# -------------------------------------------------------
# 5) Training function (simple)
# -------------------------------------------------------
def train_autoencoder(model, dataloader, n_epochs=5, lr=1e-3, verbose=True):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    for epoch in range(n_epochs):
        model.train()
        running = 0.0
        for xb, _ in dataloader:
            xb = xb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = criterion(out, xb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        epoch_loss = running / len(dataloader.dataset)
        if verbose:
            print(f"Epoch {epoch+1}/{n_epochs} - loss: {epoch_loss:.6f}")
    return model

# -------------------------------------------------------
# 6) Visualization helpers: grid, diffs, single sample
# -------------------------------------------------------
def plot_grid(originals, reconstructions, n_rows=4, n_cols=8, show_diff=False, cmap='gray'):
    # originals, reconstructions: numpy arrays shape (N, H, W)
    N = min(len(originals), n_rows*n_cols)
    fig, axes = plt.subplots(2*n_rows, n_cols, figsize=(n_cols*1.6, n_rows*1.6*2))
    axes = axes.reshape(2*n_rows, n_cols)
    idx = 0
    for r in range(n_rows):
        for c in range(n_cols):
            if idx < N:
                axes[2*r, c].imshow(originals[idx], cmap=cmap)
                axes[2*r, c].axis('off')
                axes[2*r, c].set_title(f"orig {idx}", fontsize=8)
                axes[2*r+1, c].imshow(reconstructions[idx], cmap=cmap)
                axes[2*r+1, c].axis('off')
                axes[2*r+1, c].set_title(f"recon {idx}", fontsize=8)
                if show_diff:
                    # overlay difference as title value (MSE)
                    mse = np.mean((originals[idx]-reconstructions[idx])**2)
                    axes[2*r+1, c].set_xlabel(f"MSE:{mse:.4f}", fontsize=8)
            else:
                axes[2*r, c].axis('off')
                axes[2*r+1, c].axis('off')
            idx += 1
    plt.tight_layout()
    plt.show()

def single_sample_display(orig, recon, diff_scale=1.0):
    # orig/recon are HxW
    fig, axs = plt.subplots(1,3, figsize=(9,3))
    axs[0].imshow(orig, cmap='gray'); axs[0].set_title('Original'); axs[0].axis('off')
    axs[1].imshow(recon, cmap='gray'); axs[1].set_title('Reconstrucción'); axs[1].axis('off')
    diff = np.abs(orig - recon)
    axs[2].imshow(diff, cmap='inferno'); axs[2].set_title('Diferencia (abs)'); axs[2].axis('off')
    plt.show()

# -------------------------------------------------------
# 7) Interactive UI: Entrenar rápido y mostrar grid
# -------------------------------------------------------
# Widgets: dataset, epochs, grid rows/cols, sample seed, show diff, train button
dataset_w = widgets.Dropdown(options=['FashionMNIST','MNIST'], value='FashionMNIST', description='Dataset')
epochs_w = widgets.IntSlider(value=4, min=1, max=20, step=1, description='Epochs')
latent_w = widgets.IntSlider(value=64, min=8, max=512, step=8, description='Latent')
rows_w = widgets.IntSlider(value=4, min=1, max=8, step=1, description='Rows')
cols_w = widgets.IntSlider(value=8, min=1, max=10, step=1, description='Cols')
seed_w = widgets.IntSlider(value=42, min=0, max=9999, step=1, description='Seed')
showdiff_w = widgets.Checkbox(value=False, description='Mostrar MSE')
train_btn = widgets.Button(description='Entrenar & Mostrar Grid', button_style='success')
inspect_idx = widgets.IntSlider(value=0, min=0, max=99, step=1, description='Índice')

out = widgets.Output()

def on_train_clicked(b):
    with out:
        clear_output(wait=True)
        # reproducibility
        seed = seed_w.value
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        # load dataset
        ds = get_dataset(dataset_w.value, train=True)
        # create loader for training
        loader = DataLoader(ds, batch_size=256, shuffle=True, num_workers=2)
        # model
        latent_dim = latent_w.value
        model = ConvAutoencoder(latent_dim=latent_dim).to(device)
        print(f"Entrenando AE conv en {dataset_w.value}  (latent={latent_dim}) ...")
        train_autoencoder(model, loader, n_epochs=epochs_w.value, lr=1e-3, verbose=True)
        # Prepare sample set to display (take first N of test set)
        ds_test = get_dataset(dataset_w.value, train=False)
        X = torch.stack([ds_test[i][0] for i in range(len(ds_test))])  # shape (N,1,28,28)
        # compute reconstructions in batches
        model.eval()
        recon_all = []
        orig_all = []
        batch_size = 256
        with torch.no_grad():
            for i in range(0, len(X), batch_size):
                xb = X[i:i+batch_size].to(device)
                outp = model(xb).cpu().numpy()  # (b,1,28,28)
                recon_all.append(outp)
                orig_all.append(xb.cpu().numpy())
        recon_all = np.concatenate(recon_all, axis=0)
        orig_all = np.concatenate(orig_all, axis=0)
        # squeeze channel dim and make HxW
        recon_imgs = recon_all.squeeze(1)
        orig_imgs = orig_all.squeeze(1)
        n_rows = rows_w.value
        n_cols = cols_w.value
        print("Mostrando grid (original arriba, reconstrucción abajo):")
        plot_grid(orig_imgs, recon_imgs, n_rows=n_rows, n_cols=n_cols, show_diff=showdiff_w.value)
        # update inspect idx max
        inspect_idx.max = max(0, len(orig_imgs)-1)
        # store in widget state for later inspection
        inspect_idx.model = model  # attach
        inspect_idx.orig_imgs = orig_imgs
        inspect_idx.recon_imgs = recon_imgs

train_btn.on_click(on_train_clicked)

def on_inspect_change(change):
    # display a single example for index
    if not hasattr(inspect_idx, 'orig_imgs'):
        return
    idx = inspect_idx.value
    orig = inspect_idx.orig_imgs[idx]
    recon = inspect_idx.recon_imgs[idx]
    clear_output(wait=True)
    print(f"Índice {idx} (original / reconstrucción / diferencia):")
    single_sample_display(orig, recon)

inspect_idx.observe(on_inspect_change, names='value')

# Layout
ui = widgets.VBox([
    widgets.HBox([dataset_w, epochs_w, latent_w]),
    widgets.HBox([rows_w, cols_w, seed_w, showdiff_w]),
    train_btn,
    widgets.Label("Inspeccionar un ejemplo individual:"),
    inspect_idx,
    out
])
display(ui)

# -------------------------------------------------------
# 8) Explicaciones en el notebook (texto útil para presentar)
# -------------------------------------------------------
print("\n--- Pautas para explicar el resultado ---")
print("1) Muestra el grid: explica qué fila superior (original) vs fila inferior (reconstrucción).")
print("2) Señala ejemplos donde se pierden detalles y plantea hipótesis: 'se perdió la textura del tejido' o 'se confundió la forma' - relaciona esto con el tamaño del latent y la profundidad del modelo.")
print("3) Explica el rol del MSE (qué mide) y sus limitaciones (no necesariamente coincide con calidad perceptual).")
print("4) Para mejorar, plantea experimentos: aumentar latent, usar denoising, añadir capas o entrenar más tiempo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.5 MB/s eta 0:00:00
Device: cuda



--- Pautas para explicar el resultado ---
1) Muestra el grid: explica qué fila superior (original) vs fila inferior (reconstrucción).
2) Señala ejemplos donde se pierden detalles y plantea hipótesis: 'se perdió la textura del tejido' o 'se confundió la forma' - relaciona esto con el tamaño del latent y la profundidad del modelo.
3) Explica el rol del MSE (qué mide) y sus limitaciones (no necesariamente coincide con calidad perceptual).
4) Para mejorar, plantea experimentos: aumentar latent, usar denoising, añadir capas o entrenar más tiempo.


In [3]:
# Interactive latent sliders -> real-time reconstruction (Colab-ready)
# Pega esto al final de tu notebook y ejecútalo.
# Requiere ipywidgets, matplotlib y PyTorch (deberían estar ya instalados si seguiste las celdas previas).

import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipywidgets import Layout
from torch.utils.data import DataLoader

# ---------- Fallbacks: si no existe 'model' o 'get_dataset', crea/entrena uno rapido ----------
need_demo = False
try:
    model  # check existence
    get_dataset  # check existence
except Exception:
    need_demo = True

if need_demo:
    print("No se detectó 'model' o 'get_dataset' en el entorno. Se creará y entrenará un AE demo en FashionMNIST (rápido).")
    from torchvision import datasets, transforms
    # define get_dataset simple
    def get_dataset(name='FashionMNIST', train=True):
        transform = transforms.Compose([transforms.ToTensor()])
        if name == 'FashionMNIST':
            return datasets.FashionMNIST(root='./data', train=train, download=True, transform=transform)
        elif name == 'MNIST':
            return datasets.MNIST(root='./data', train=train, download=True, transform=transform)
        else:
            raise ValueError("Dataset desconocido")
    # simple conv AE (same structure as previously used)
    class ConvAutoencoder(torch.nn.Module):
        def __init__(self, latent_dim=32):
            super().__init__()
            self.encoder = torch.nn.Sequential(
                torch.nn.Conv2d(1, 16, 3, stride=2, padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(16, 32, 3, stride=2, padding=1),
                torch.nn.ReLU(),
                torch.nn.Flatten(),
                torch.nn.Linear(32*7*7, latent_dim)
            )
            self.decoder = torch.nn.Sequential(
                torch.nn.Linear(latent_dim, 32*7*7),
                torch.nn.ReLU(),
                torch.nn.Unflatten(1, (32,7,7)),
                torch.nn.ConvTranspose2d(32,16,3,stride=2,padding=1,output_padding=1),
                torch.nn.ReLU(),
                torch.nn.ConvTranspose2d(16,1,3,stride=2,padding=1,output_padding=1),
                torch.nn.Sigmoid()
            )
        def forward(self,x):
            z = self.encoder(x)
            return self.decoder(z)
        def encode(self,x):
            return self.encoder(x)
        def decode(self,z):
            return self.decoder(z)

    # train quick demo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ds_train = get_dataset('FashionMNIST', train=True)
    loader_tr = DataLoader(ds_train, batch_size=256, shuffle=True, num_workers=2)
    model = ConvAutoencoder(latent_dim=32).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = torch.nn.MSELoss()
    for epoch in range(3):
        model.train()
        running = 0.0
        for xb, _ in loader_tr:
            xb = xb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = loss_fn(out, xb)
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        print(f"Demo train epoch {epoch+1} loss: {running/len(loader_tr.dataset):.6f}")
    print("Modelo demo entrenado. Puedes cambiar dataset a MNIST si lo deseas.")

# ---------- Preparar dataset y estadísticas latente ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# dataset selector
dataset_dropdown = widgets.Dropdown(options=['FashionMNIST','MNIST'], value='FashionMNIST', description='Dataset:')
# load datasets
from torchvision import transforms, datasets
def load_dataset(name, train):
    transform = transforms.Compose([transforms.ToTensor()])
    if name == 'FashionMNIST':
        return datasets.FashionMNIST(root='./data', train=train, download=True, transform=transform)
    else:
        return datasets.MNIST(root='./data', train=train, download=True, transform=transform)

# compute latent stats (mu,std) using up to max_samples
def compute_latent_stats(model, dataset, max_samples=2000, batch_size=512):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    zs = []
    seen = 0
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            z = model.encode(xb)  # (B, latent_dim)
            zs.append(z.cpu().numpy())
            seen += z.shape[0]
            if seen >= max_samples:
                break
    zs = np.concatenate(zs, axis=0)[:max_samples]
    mu = zs.mean(axis=0)
    sigma = zs.std(axis=0)
    return mu, sigma

# decode helper
def decode_from_z(model, z_np):
    model.eval()
    with torch.no_grad():
        z_t = torch.tensor(z_np, dtype=torch.float32).to(device)
        recon = model.decode(z_t).cpu().numpy()  # (B,1,H,W) or (B, H*W)
    if recon.ndim == 2:
        side = int(np.sqrt(recon.shape[1]))
        recon = recon.reshape((-1,1,side,side))
    return recon[:,0,:,:]  # return (B,H,W)

# ---------- UI: sliders for each latent dimension + reconstruction display ----------
# parameters
MAX_SLIDERS_DISPLAY = 200  # safety cap

# widgets: sample index, reset, apply, info
sample_idx = widgets.IntSlider(value=0, min=0, max=9999, step=1, description='Sample idx:')
reset_btn = widgets.Button(description='Reset to encoded z', button_style='info')
apply_btn = widgets.Button(description='Apply current sliders', button_style='success')
dataset_info = widgets.HTML(value="")  # show dataset id/info

# Container placeholders
sliders_box = widgets.VBox(layout=Layout(overflow='auto', height='420px', width='46%'))  # left side
out_box = widgets.Output(layout=Layout(border='1px solid #ddd', padding='10px', width='52%'))   # right side

main_box = widgets.HBox([sliders_box, out_box], layout=Layout(width='100%'))

# global state for sliders
current_sliders = []
current_mu = None
current_sigma = None
current_z0 = None
current_latent_dim = None
current_dataset_name = None
current_test_dataset = None

# function to (re)build sliders given latent dim and stats
def build_sliders(latent_dim, mu, sigma, z0, sigma_scale=3.0):
    global current_sliders
    current_sliders = []
    children = []
    for i in range(latent_dim):
        s = sigma[i] if sigma[i] > 1e-6 else 1.0
        low = float(mu[i] - sigma_scale*s)
        high = float(mu[i] + sigma_scale*s)
        # if latent_dim large, keep narrow step
        step = (high - low) / 100.0 if (high-low) > 0 else 0.1
        fs = widgets.FloatSlider(value=float(z0[i]), min=low, max=high, step=step,
                                 description=f"z[{i}]", continuous_update=False,
                                 layout=Layout(width='95%'))
        current_sliders.append(fs)
        children.append(fs)
    # arrange children in VBox (scrollable)
    sliders_box.children = children
    return current_sliders

# update output display (reconstruction + original + info)
def update_output_display():
    with out_box:
        clear_output(wait=True)
        # assemble z from sliders
        z_vals = np.array([s.value for s in current_sliders], dtype=np.float32).reshape(1, -1)
        recon = decode_from_z(model, z_vals)[0]  # HxW
        # original sample
        ds = current_test_dataset
        idx = min(sample_idx.value, len(ds)-1)
        orig, _ = ds[idx]
        orig_np = orig.numpy().squeeze(0)
        # plot side by side
        fig, axes = plt.subplots(1,2, figsize=(6.5,3.5))
        axes[0].imshow(orig_np, cmap='gray'); axes[0].set_title(f"Original (idx={idx})"); axes[0].axis('off')
        axes[1].imshow(recon, cmap='gray'); axes[1].set_title("Reconstrucción (modificada)"); axes[1].axis('off')
        plt.show()
        # show brief latent info
        z_vals_flat = z_vals.flatten()
        # show first 8 dimension values as text
        display(widgets.HTML(value=f"<b>Latent dim:</b> {len(z_vals_flat)} &nbsp;&nbsp; <b>Dataset:</b> {current_dataset_name}"))
        # optionally show vector snippet
        snippet = ", ".join([f"{v:.2f}" for v in z_vals_flat[:8]])
        if len(z_vals_flat) > 8:
            snippet += " ..."
        display(widgets.HTML(value=f"<b>z[0..7]:</b> {snippet}"))
        # dataset id / hint
        display(widgets.HTML(value=f"<i>Visualización: imagen original y reconstrucción con las dimensiones latentes ajustadas manualmente.</i>"))

# handler: when sample index changes -> recompute z0 and reset sliders default values
def on_sample_change(change):
    idx = min(sample_idx.value, len(current_test_dataset)-1)
    sample, _ = current_test_dataset[idx]
    sample_t = sample.unsqueeze(0).to(device)
    with torch.no_grad():
        z0 = model.encode(sample_t).cpu().numpy().reshape(-1)
    # update sliders' defaults without rebuilding ranges
    if len(current_sliders) == len(z0):
        for i,s in enumerate(current_sliders):
            s.value = float(z0[i])
    else:
        # rebuild with proper sizes
        build_sliders(len(z0), current_mu, current_sigma, z0)
    update_output_display()

sample_idx.observe(on_sample_change, names='value')

# reset to encoded z
def on_reset_clicked(b):
    idx = min(sample_idx.value, len(current_test_dataset)-1)
    sample, _ = current_test_dataset[idx]
    sample_t = sample.unsqueeze(0).to(device)
    with torch.no_grad():
        z0 = model.encode(sample_t).cpu().numpy().reshape(-1)
    for i,s in enumerate(current_sliders):
        s.value = float(z0[i])
    update_output_display()

reset_btn.on_click(on_reset_clicked)
apply_btn.on_click(lambda b: update_output_display())

# slider change: update live (debounced by continuous_update=False for responsiveness)
def slider_changed(change):
    # simply update output display when any slider reports change
    update_output_display()

# connect sliders later after building them

# Build initial stats and sliders for default dataset/sample
def initialize_ui(dataset_name):
    global current_mu, current_sigma, current_z0, current_latent_dim, current_dataset_name, current_test_dataset
    current_dataset_name = dataset_name
    ds_train = load_dataset(dataset_name, train=True)
    ds_test = load_dataset(dataset_name, train=False)
    current_test_dataset = ds_test
    # compute mu/sigma (may take a few seconds)
    print("Computando estadísticas latentes (mu/std) — esto puede tardar unos segundos...)")
    mu, sigma = compute_latent_stats(model, ds_train, max_samples=2000)
    current_mu = mu
    current_sigma = sigma
    # sample z0
    idx = min(sample_idx.value, len(ds_test)-1)
    sample, _ = ds_test[idx]
    sample_t = sample.unsqueeze(0).to(device)
    with torch.no_grad():
        z0 = model.encode(sample_t).cpu().numpy().reshape(-1)
    current_z0 = z0
    current_latent_dim = z0.shape[0]
    # safety cap
    if current_latent_dim > MAX_SLIDERS_DISPLAY:
        print(f"Advertencia: latent_dim = {current_latent_dim} (muy grande). Se mostrarán los primeros {MAX_SLIDERS_DISPLAY} sliders. Puedes ampliar el código si quieres todos.")
        mu = mu[:MAX_SLIDERS_DISPLAY]
        sigma = sigma[:MAX_SLIDERS_DISPLAY]
        z0 = z0[:MAX_SLIDERS_DISPLAY]
    # build sliders and observe changes
    build_sliders(len(z0), mu, sigma, z0)
    for s in current_sliders:
        s.observe(slider_changed, names='value')
    # adjust sample_idx max
    sample_idx.max = len(ds_test)-1
    # show initial output
    update_output_display()

# dataset selection change
def on_dataset_change(change):
    initialize_ui(change['new'])

dataset_dropdown.observe(on_dataset_change, names='value')

# initialize UI with default dataset
initialize_ui(dataset_dropdown.value)

# assemble controls UI
controls_row = widgets.HBox([dataset_dropdown, sample_idx, reset_btn, apply_btn])
display(controls_row)
display(main_box)

# Info note
print("Instrucciones:")
print("- Ajusta los sliders (cada uno corresponde a una dimensión latente) para modificar la reconstrucción.")
print("- 'Reset to encoded z' vuelve al z codificado del ejemplo seleccionado.")
print("- Si tu latent_dim es grande, la caja de sliders es scrollable; puedes ajustar el código para mostrar otros subconjuntos si lo necesitas.")

Computando estadísticas latentes (mu/std) — esto puede tardar unos segundos...)


Instrucciones:
- Ajusta los sliders (cada uno corresponde a una dimensión latente) para modificar la reconstrucción.
- 'Reset to encoded z' vuelve al z codificado del ejemplo seleccionado.
- Si tu latent_dim es grande, la caja de sliders es scrollable; puedes ajustar el código para mostrar otros subconjuntos si lo necesitas.
